In [1]:
import os
import time
from google import genai
from google.genai import types
from dotenv import load_dotenv
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, hamming_loss


from datasets import load_dataset

In [2]:
ds = load_dataset("Rami/multi-label-class-github-issues-text-classification")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 778 entries, 0 to 777
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     778 non-null    object
 1   labels    778 non-null    object
 2   bodyText  778 non-null    object
dtypes: object(3)
memory usage: 18.4+ KB


In [3]:
labels = ["bug", "feature", "question", "won't fix", "docs"]

test = test[test['labels'].apply(lambda cats: all(c in labels for c in cats))]
test = test[test['labels'].apply(len) > 0]
test.rename(columns={'title': 'text'}, inplace=True)
test.drop(columns=['bodyText'], inplace=True)
test.reset_index(drop=True, inplace=True)

test

,text,labels
0,Update CONTRIBUTING.md on bugfixes/features PRs,[docs]
1,How to print the metric (across all working tr...,"[question, won't fix]"
2,Model loaded from checkpoint has bad accuracy,[question]
3,Add a robots.txt to stop Google from indexing ...,[docs]
4,Multi-processing with IterableDataset Warning,[docs]
...,...,...
195,TensorBoardLogger and ModelCheckpoint are not ...,[bug]
196,imagenet_example cannot run,[bug]
197,log_gpu_memory='all'` options raise Error,[bug]
198,`overfit_pct` vs `train_percent_check` etc,"[feature, docs]"


In [4]:
mlb = MultiLabelBinarizer()
test_labels_binarized = mlb.fit_transform(test['labels'])

test_labels_df = pd.DataFrame(test_labels_binarized, columns=mlb.classes_)

test = pd.concat([test, test_labels_df], axis=1)

test.drop(columns=['labels'], inplace=True)

In [5]:
load_dotenv()

api_key=os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

In [6]:
def classify(text, labels):

    sys_instruct="You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling multilabel classification tasks based on user instructions."

    start_time = time.time()

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        config=types.GenerateContentConfig(
            system_instruction=sys_instruct,
            safety_settings=[
            types.SafetySetting(
                category="HARM_CATEGORY_HARASSMENT",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_HATE_SPEECH",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_DANGEROUS_CONTENT",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_CIVIC_INTEGRITY",
                threshold="BLOCK_NONE"
            ),
            ],
        ),
        contents=f"Classify the following text based on the task: Classification of github issues. Only respond with the labels that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"
    )

    request_time = time.time() - start_time
    completion = response.text
    if completion:
        completion = completion.lower()
    else:
        completion = "None"
    completion_tokens = response.usage_metadata.candidates_token_count
    prompt_tokens = response.usage_metadata.prompt_token_count
    total_tokens = response.usage_metadata.total_token_count

    return completion, request_time, completion_tokens, prompt_tokens, total_tokens

def post_process(text):
    list = []
    if 'bug' in text:
        list.append('bug')
    if 'feature' in text:
        list.append('feature')
    if 'question' in text:
        list.append('question')
    if "won't fix" in text:
        list.append("won't fix")
    if 'docs' in text:
        list.append('docs')

    return list

In [7]:
pred_df = test.copy() 

for index, row in pred_df.iterrows():
    try:
        text = row['text']
        completion, request_time, completion_tokens, prompt_tokens, total_tokens = classify(text, labels)
        pred_df.at[index, 'prediction'] = completion
        pred_df.at[index, 'request_time'] = request_time
        pred_df.at[index, 'completion_tokens'] = completion_tokens
        pred_df.at[index, 'prompt_tokens'] = prompt_tokens
        pred_df.at[index, 'total_tokens'] = total_tokens

    except Exception as e:
        # Save the current state of the DataFrame to a file before breaking out or retrying.
        pred_df.to_csv("results/partial_gemini_ZS_multilabel2.csv", index=False)
        print(f"An error occurred at index {index}: {e}. Partial results saved.")
        # Optionally, you can break out of the loop or continue based on your needs.
        break

pred_df['prediction_post_processed'] = pred_df['prediction'].apply(post_process)
pred_df.to_csv("results/gemini_ZS_multilabel2.csv", index=False)

pred_df

,text,bug,docs,feature,question,won't fix,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
0,Update CONTRIBUTING.md on bugfixes/features PRs,0,1,0,0,0,docs\n,1.169045,2.0,96.0,98.0,[docs]
1,How to print the metric (across all working tr...,0,0,0,1,1,question\n,0.510695,2.0,107.0,109.0,[question]
2,Model loaded from checkpoint has bad accuracy,0,0,0,1,0,bug\n,1.274131,2.0,92.0,94.0,[bug]
3,Add a robots.txt to stop Google from indexing ...,0,1,0,0,0,docs\n,1.084701,2.0,99.0,101.0,[docs]
4,Multi-processing with IterableDataset Warning,0,1,0,0,0,"bug, question\n",1.103902,4.0,92.0,96.0,"[bug, question]"
...,...,...,...,...,...,...,...,...,...,...,...,...
195,TensorBoardLogger and ModelCheckpoint are not ...,1,0,0,0,0,"bug, feature\n",1.136301,4.0,98.0,102.0,"[bug, feature]"
196,imagenet_example cannot run,1,0,0,0,0,bug\n,1.089096,2.0,90.0,92.0,[bug]
197,log_gpu_memory='all'` options raise Error,1,0,0,0,0,bug\n,1.058695,2.0,95.0,97.0,[bug]
198,`overfit_pct` vs `train_percent_check` etc,0,1,1,0,0,"bug, question\n",0.578453,4.0,99.0,103.0,"[bug, question]"


In [8]:
for label in labels:
    pred_df[f"{label} pred"] = pred_df.apply(lambda row: 1 if label in row['prediction_post_processed'] else 0, axis=1)

pred_df = pred_df.drop(columns=['prediction'])

pred_df

,text,bug,docs,feature,question,won't fix,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed,bug pred,feature pred,question pred,won't fix pred,docs pred
0,Update CONTRIBUTING.md on bugfixes/features PRs,0,1,0,0,0,1.169045,2.0,96.0,98.0,[docs],0,0,0,0,1
1,How to print the metric (across all working tr...,0,0,0,1,1,0.510695,2.0,107.0,109.0,[question],0,0,1,0,0
2,Model loaded from checkpoint has bad accuracy,0,0,0,1,0,1.274131,2.0,92.0,94.0,[bug],1,0,0,0,0
3,Add a robots.txt to stop Google from indexing ...,0,1,0,0,0,1.084701,2.0,99.0,101.0,[docs],0,0,0,0,1
4,Multi-processing with IterableDataset Warning,0,1,0,0,0,1.103902,4.0,92.0,96.0,"[bug, question]",1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,TensorBoardLogger and ModelCheckpoint are not ...,1,0,0,0,0,1.136301,4.0,98.0,102.0,"[bug, feature]",1,1,0,0,0
196,imagenet_example cannot run,1,0,0,0,0,1.089096,2.0,90.0,92.0,[bug],1,0,0,0,0
197,log_gpu_memory='all'` options raise Error,1,0,0,0,0,1.058695,2.0,95.0,97.0,[bug],1,0,0,0,0
198,`overfit_pct` vs `train_percent_check` etc,0,1,1,0,0,0.578453,4.0,99.0,103.0,"[bug, question]",1,0,1,0,0


In [9]:
y_true = pred_df[labels].values
y_pred = pred_df[[f"{label} pred" for label in labels]].values

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)
hamming_loss = hamming_loss(y_true, y_pred)
print('Hamming loss: %f' % hamming_loss)

Accuracy: 0.430000
F1 score: 0.590893
Precision: 0.611975
Recall: 0.611336
Hamming loss: 0.191000


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [10]:
# get average response time, vram usage and ram usage
request_time_avg = pred_df['request_time'].mean()
completion_tokens_avg = pred_df['completion_tokens'].mean()
prompt_tokens_avg = pred_df['prompt_tokens'].mean()
total_tokens_avg = pred_df['total_tokens'].mean()

print(f'Average response time: {request_time_avg}')
print(f'Average completion tokens: {completion_tokens_avg}')
print(f'Average prompt tokens: {prompt_tokens_avg}')
print(f'Average total tokens: {total_tokens_avg}')

Average response time: 1.2154878401756286
Average completion tokens: 2.46
Average prompt tokens: 94.865
Average total tokens: 97.325


In [11]:
input_token_price = 0.1/1_000_000
output_token_price = 0.4/1_000_000

# Calculate the cost of the requests
total_cost = 0
for index, row in pred_df.iterrows():
    completion_tokens = row['completion_tokens']
    prompt_tokens = row['prompt_tokens']
    cost  = completion_tokens * output_token_price + prompt_tokens * input_token_price
    total_cost += cost

print(f'Total cost: USD {total_cost}')

Total cost: USD 0.0020941000000000006


In [12]:
with open('results/gemini_ZS_multilabel2.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Hamming loss: {hamming_loss}\n')
    f.write(f'Average response time: {request_time_avg}\n')
    f.write(f'Average completion tokens: {completion_tokens_avg}\n')
    f.write(f'Average prompt tokens: {prompt_tokens_avg}\n')
    f.write(f'Average total tokens: {total_tokens_avg}\n')
    f.write(f'Total cost: USD {total_cost}\n')